# Energy flows under the CO2 tax with the technology block (DREAM's S3)

DREAM's S3: CO2 tax +750, phased in 2025-2030, with the energy technology block on. Solved by ModelFlow from
DREAM's staged start and equal to DREAM's solution (max relative difference 6e-7). All flows are for 2030, when
the tax is fully phased in.

* **Energy**: `qEpj[es,e,d]`, energy use in PJ by energy good `e`, purpose `es` and user `d`.
* **Fossil CO2**: `qEmmE_BU[co2ubio,es,e,d]`, bottom-up CO2 from fossil fuels along the same chain (thousand
  tonnes, including bunkering abroad, so the total is larger than the national inventory).

Energy goods are grouped into 8 groups and users into 11 groups (`tools/greu_viz.py`). The Sankey diagrams
show positive flows only: captured CO2 (CCS) enters the model as a negative energy flow, and it is shown in the
change table below instead. Flows under 0.2 % of the total are left out of the diagrams.

*Example calculations based on example data. The model is GREU (ModelFlow translation), not DREAM's GreenREFORM-EU.*

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'tools'))
import greu_viz as gv
pd.set_option('display.width', 200)

In [2]:
Y = 2030
base = gv.baseline('main_energy_technology')
shock = gv.solution('shock_co2tax_et')
fb, fs = gv.flows(base, Y), gv.flows(shock, Y)
print(f'energy use {Y}: baseline {fb.value.sum():,.0f} PJ, CO2 tax {fs.value.sum():,.0f} PJ '
      f'({gv.pct(fs.value.sum(), fb.value.sum()):.1f} %)')

energy use 2030: baseline 2,211 PJ, CO2 tax 2,105 PJ (-4.8 %)


## Energy: baseline

Oil products dominate, mostly as transport fuel. Bunkering of Danish ships, planes and trucks abroad is part
of oil products.

In [3]:
gv.sankey(fb, f'Energy use {Y}, baseline (PJ): energy good → purpose → user', 'PJ').show()

## Energy: with the CO2 tax

In [4]:
gv.sankey(fs, f'Energy use {Y}, CO2 tax +750 (PJ): energy good → purpose → user', 'PJ').show()

## What changes

Change in energy use by energy good and purpose (PJ), CO2 tax against the baseline. Blue is less, red is more.
Captured CO2 is negative in both runs; a more negative number means more CO2 captured.

In [5]:
t = gv.change_table(fb, fs, ('good', 'purpose'))
mat = t['change'].unstack('purpose').reindex(index=gv.GOOD_GROUPS).dropna(how='all')
mat = mat[[p for p in gv.PURPOSE.values() if p in mat.columns]]
lim = np.nanmax(np.abs(mat.values))
fig = go.Figure(go.Heatmap(z=mat.values, x=mat.columns, y=mat.index, zmid=0, zmin=-lim, zmax=lim,
                           colorscale=[[0, '#1c5cab'], [0.5, '#f0efec'], [1, '#e34948']],
                           text=np.where(np.isfinite(mat.values), np.round(mat.values, 1), ''),
                           texttemplate='%{text}', textfont=dict(size=11),
                           hovertemplate='%{y}, %{x}: %{z:.2f} PJ<extra></extra>', xgap=2, ygap=2,
                           colorbar=dict(title='PJ')))
fig.update_layout(template=gv.TEMPLATE, height=420, title=f'Change in energy use {Y}, PJ (CO2 tax − baseline)')
fig.update_yaxes(autorange='reversed')
fig.show()
gv.change_table(fb, fs, ('good',)).round(2)

,baseline,shock,change,% change
good,,,,
Bioenergy and waste,198.67,195.76,-2.90,-1.46
Captured CO2,-25.85,-93.06,-67.21,260.02
Coal,30.75,30.07,-0.68,-2.20
District heat,127.79,127.77,-0.02,-0.01
Electricity,213.08,208.95,-4.12,-1.93
Natural gas,292.57,287.55,-5.03,-1.72
Oil products,1373.94,1347.61,-26.33,-1.92


## Fossil CO2: baseline and with the tax

In [6]:
eb = gv.flows(base, Y, 'qemme_bu', em='co2ubio')
es = gv.flows(shock, Y, 'qemme_bu', em='co2ubio')
print(f'fossil CO2 {Y} (bottom-up, incl. bunkering): baseline {eb.value.sum():,.0f}, CO2 tax {es.value.sum():,.0f} '
      f'({gv.pct(es.value.sum(), eb.value.sum()):.1f} %)')
gv.sankey(eb, f'Fossil CO2 {Y}, baseline (kt): energy good → purpose → user', 'kt').show()

fossil CO2 2030 (bottom-up, incl. bunkering): baseline 66,501, CO2 tax 64,131 (-3.6 %)


In [7]:
gv.sankey(es, f'Fossil CO2 {Y}, CO2 tax +750 (kt): energy good → purpose → user', 'kt').show()
gv.change_table(eb, es, ('user',)).sort_values('change').round(1)

,baseline,shock,change,% change
user,,,,
Transport industries,43128.9,42578.1,-550.8,-1.3
"Households, cars",5090.5,4563.3,-527.2,-10.4
Manufacturing,3173.2,2892.9,-280.3,-8.8
Services,3325.7,3070.0,-255.7,-7.7
Construction,1652.5,1424.5,-228.0,-13.8
"Agriculture, forestry, fishing",1515.8,1317.2,-198.7,-13.1
"Households, housing",1637.5,1475.3,-162.2,-9.9
Public services,788.7,687.2,-101.5,-12.9
Energy supply and extraction,6188.0,6122.7,-65.3,-1.1


## Energy prices

Average price paid per PJ for each energy good group (value `vEpj` over quantity `qEpj`, summed over purposes
and users), % change against the baseline.

In [8]:
vn, qn = gv.names_of('vepj'), gv.names_of('qepj')
def price_by_group(df, y):
    v = pd.Series({gv.GOOD_OF[r[1]]: 0.0 for r in qn.values()})
    q = v.copy()
    for c, (es_, e, d) in qn.items():
        vc = c.replace('QEPJ__', 'VEPJ__', 1)
        if vc in df and df.at[y, c] > 0:
            v[gv.GOOD_OF[e]] += df.at[y, vc]; q[gv.GOOD_OF[e]] += df.at[y, c]
    return v / q
p = gv.pct(price_by_group(shock, Y), price_by_group(base, Y)).reindex(gv.GOOD_GROUPS).dropna()
p = p.drop('Captured CO2', errors='ignore').sort_values()
fig = go.Figure(go.Bar(x=p.values, y=p.index, orientation='h', marker=dict(color=gv.CAT[0]),
                       text=[f'{v:.1f} %' for v in p.values], textposition='outside', textfont=dict(color='#52514e'),
                       hovertemplate='%{y}: %{x:.2f} %<extra></extra>'))
fig.update_layout(template=gv.TEMPLATE, height=360, title=f'Average energy price {Y}, % change', xaxis_title='%')
fig.show()

## Technology shares

The energy technology block chooses technologies for two energy services: heating in the dairy industry
(10030) and processes under the ETS in cement (23001). Each technology is labelled by its energy inputs. The
bars show the change in share in 2030 (percentage points), the lines the technologies that move, over time
(dashed: baseline, solid: with the tax).

In [9]:
units = {'HEATING__10030': 'Heating, dairy products (10030)', 'IN_ETS__23001': 'Processes in the ETS, cement (23001)'}
fig = make_subplots(rows=1, cols=2, subplot_titles=list(units.values()), horizontal_spacing=0.32)
movers = {}
for j, (u, title) in enumerate(units.items(), 1):
    lab = gv.technology_labels(base, u)
    sb, ss = gv.shares(base, u), gv.shares(shock, u)
    d = (100 * (ss.loc[Y] - sb.loc[Y])).sort_values()
    movers[u] = list(d.abs().sort_values(ascending=False).index[:4])
    fig.add_trace(go.Bar(x=d.values, y=[lab.get(l, l) for l in d.index], orientation='h', showlegend=False,
                         marker=dict(color=['#1c5cab' if v < 0 else '#e34948' for v in d.values]),
                         hovertemplate='%{y}: %{x:.2f} pp<extra></extra>'), row=1, col=j)
fig.update_layout(template=gv.TEMPLATE, height=420, title=f'Change in technology shares {Y}, percentage points')
fig.show()

In [10]:
u = 'IN_ETS__23001'
lab = gv.technology_labels(base, u)
sb, ss = gv.shares(base, u), gv.shares(shock, u)
fig = go.Figure()
for k, l in enumerate(movers[u]):
    fig.add_trace(go.Scatter(x=sb.index, y=100 * sb[l], mode='lines', line=dict(color=gv.CAT[k], width=2, dash='dash'),
                             name=f'{lab[l]}, baseline', legendgroup=l, hovertemplate='%{x}: %{y:.1f} %<extra></extra>'))
    fig.add_trace(go.Scatter(x=ss.index, y=100 * ss[l], mode='lines', line=dict(color=gv.CAT[k], width=2),
                             name=f'{lab[l]}, CO2 tax', legendgroup=l, hovertemplate='%{x}: %{y:.1f} %<extra></extra>'))
fig.update_layout(template=gv.TEMPLATE, height=420, hovermode='x unified', yaxis_title='% of the energy service',
                  title=units[u] + ': the technologies that move')
fig.show()